**Técnicas de Aprendizaje Automático**

_Máster Universitario en Inteligencia Artificial_

# Laboratorio: Regresión lineal y árboles de decisión para tareas de regresión


## Objetivos

Mediante esta actividad se pretende que el alumno ponga en práctica los pasos para la resolución de un problema de machine learning, el tratamiento de datos y la creación de modelos basados en regresión lineal y árboles de decisión. El objetivo es comprender de forma práctica con un problema determinado las diferencias que existen a la hora de entrenar los diferentes modelos.

- Iniciarse en el Análisis Exploratorio de Datos (EDA) para los problemas de Machine Learning.
- Entender y aplicar los conceptos de la Regresión Lineal Múltiple a un problema de **regresión**.
- Entender y aplicar los conceptos de Árboles de Decisión a un problema de **regresión**.
- Evaluar y analizar los resultados de los clasificadores.
- Investigar la aplicación de los modelos de **regresión** a problemas reales.


## Descripción de la actividad

Debes completar los espacios indicados en el notebook con el código solicitado y la respuesta en texto, en función de lo que se solicite. Ten en cuenta que las celdas vacías indican cuántas líneas debe ocupar dicha respuesta, por lo general no más de una línea.

El conjunto de datos con el que vamos a trabajar se encuentra en el siguiente enlace: https://archive.ics.uci.edu/dataset/360/air+quality

Se trata de un conjunto de datos (dataset) sobre calidad del aire. En total cuenta con 9358 instancias de respuestas promediadas por hora de una matriz de 5 sensores químicos de óxido de metal integrados en un dispositivo multisensor químico de calidad del aire. El dispositivo estaba ubicado en un área significativamente contaminada, al nivel de la carretera, dentro de una ciudad italiana. Los datos se registraron desde marzo de 2004 hasta febrero de 2005 (un año).

El objetivo de la regresión será predecir la calidad del aire para un determinado día.

### Tareas que se deben realizar

- Análisis descriptivo de los datos:
   - Debe completarse el código solicitado y responder a las preguntas. Todo ello en el notebook dado como base.
- Regresión:
  - Debe completarse el código solicitado y responder a las preguntas. Todo ello en el notebook dado como base.
- Investigación:
  - Buscar un artículo científico (https://scholar.google.es/) con un caso de uso de **regresión** empleando una de las dos técnicas (o ambas) vistas en la actividad. Los artículos deben estar en revistas científicas, y deben ser posteriores a 2015. No debe utilizar técnicas de Deep Learning.
  - Para el artículo indicar:
    - Objetivo: cuál es el objetivo de la investigación, es decir a qué problema real está aplicando la regresión.
    - Cómo utilizan las técnicas de regresión, si realizan alguna adaptación de los algoritmos indicarse.
    - Principales resultados de la aplicación y de la investigación.


### Análisis descriptivo de los datos
A continuación vas a encontrar una serie de preguntas que tendrás que responder. Para responder tendrás que escribir (y ejecutar) una (o más de una) línea de código, y a continuación indicar la respuesta en la celda indicada.

In [ ]:
## cargar el dataset
# Actividad 1 - Regresión lineal y árboles de decisión para tareas de regresión
# Dataset: Air Quality, UCI Machine Learning Repository.

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_log_error
from sklearn.impute import SimpleImputer
from pandas import DataFrame

RANDOM_STATE = 42
DATA_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00360/AirQualityUCI.zip'

# Se carga directamente desde el ZIP público de UCI. Si no hay Internet, descarga el archivo AirQualityUCI.csv
# desde UCI y déjalo en la misma carpeta del notebook.
try:
    df_raw = pd.read_csv(DATA_URL, sep=';', decimal=',', compression='zip')
except Exception:
    df_raw = pd.read_csv('AirQualityUCI.csv', sep=';', decimal=',')

# Limpieza inicial: el CSV incluye columnas vacías al final por el separador ';'.
df_raw = df_raw.dropna(axis=1, how='all')
df_raw = df_raw.dropna(axis=0, how='all')

# En este dataset los valores faltantes están codificados como -200.
df = df_raw.replace(-200, np.nan).copy()

# Para modelar se excluyen las variables temporales. Se trabaja solo con variables numéricas ambientales y de sensores.
df_model = df.drop(columns=['Date', 'Time'], errors='ignore')
df_model = df_model.apply(pd.to_numeric, errors='coerce')

print('Dimensión original:', df_raw.shape)
print('Dimensión después de limpieza básica:', df_model.shape)
df_model.head()


In [ ]:
# Vista general del conjunto de datos limpio para análisis descriptivo
df_model.info()
df_model.head()


In [ ]:
## ¿cuántas instancias tiene el dataset?

El dataset limpio para el modelado conserva las observaciones válidas del archivo original y contiene variables numéricas de concentraciones, sensores y condiciones meteorológicas. En la fuente UCI el conjunto completo contiene 9358 instancias; después de eliminar filas/columnas completamente vacías y excluir Date/Time para modelado, el número de registros debe verificarse con `df_model.shape[0]`.

In [ ]:
## ¿cuál es el tipo de datos de cada una de las columnas?
df_model.dtypes


Las variables usadas para el modelado son numéricas (`float64` después de reemplazar `-200` por `NaN`). `Date` y `Time` no se incluyen en el modelo porque son variables temporales/categóricas que requerirían ingeniería de características adicional; para esta actividad se prioriza comparar regresión lineal y árbol de decisión sobre variables ambientales continuas.

In [ ]:
## ¿cuántas columnas categóricas hay? ¿y cuántas continuas?
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
continuous_cols = df_model.select_dtypes(include=[np.number]).columns.tolist()
print('Columnas categóricas en el dataframe original:', categorical_cols)
print('Número de columnas categóricas originales:', len(categorical_cols))
print('Número de columnas continuas usadas en modelado:', len(continuous_cols))
print('Columnas continuas:', continuous_cols)


En el archivo original hay dos columnas no numéricas (`Date` y `Time`). Para el entrenamiento se trabaja con las columnas numéricas continuas, porque el objetivo es construir modelos de regresión y evitar codificar variables temporales que no son necesarias para la comparación básica de algoritmos.

In [ ]:
## ¿existen valores nulos en el dataset?
missing_summary = pd.DataFrame({
    'nulos': df_model.isna().sum(),
    'porcentaje_nulos': (df_model.isna().mean() * 100).round(2)
}).sort_values('porcentaje_nulos', ascending=False)
missing_summary


Sí existen valores faltantes. En este dataset no aparecen inicialmente como `NaN`, sino codificados con el valor `-200`, por lo que primero se reemplazaron por `NaN`. Para entrenar los modelos se eliminarán registros sin variable objetivo y se imputarán las variables predictoras con la mediana, evitando que scikit-learn falle por valores ausentes.

In [ ]:
## ¿cuál es la variable respuesta?¿de qué tipo es? (Recuerda justificar por qué la seleccionas)
target = 'C6H6(GT)'
print('Variable respuesta:', target)
print('Tipo:', df_model[target].dtype)
df_model[target].describe()


La variable respuesta seleccionada es `C6H6(GT)`, que corresponde a la concentración real horaria de benceno. Se selecciona porque es una variable continua, por tanto adecuada para regresión, y porque puede estimarse a partir de respuestas de sensores y variables ambientales. Además, al ser no negativa, permite evaluar el árbol de decisión con criterio `poisson`, como solicita la actividad.

In [ ]:
## Si te fijas en los estadísticos del dataset, ¿cómo es la distribución de las variables, CO, NOx y NO2?
pollutants = ['CO(GT)', 'NOx(GT)', 'NO2(GT)']
stats_pollutants = df_model[pollutants].describe().T
stats_pollutants['skew'] = df_model[pollutants].skew()
stats_pollutants['missing_%'] = df_model[pollutants].isna().mean() * 100
stats_pollutants


Las variables `CO(GT)`, `NOx(GT)` y `NO2(GT)` presentan distribuciones asimétricas hacia la derecha: la media suele ser mayor que la mediana y la cola superior recoge episodios de contaminación más altos. Esto es esperable en datos de calidad del aire, donde la mayoría de mediciones se concentran en valores moderados y algunos eventos elevan las concentraciones.

¿Estas variables muestran alguna distribución especial?¿Tienen datos faltantes?¿y datos anómalos?

Sí. Muestran asimetría positiva y posibles valores atípicos altos. También tienen datos faltantes, identificados originalmente con `-200`. Los valores anómalos no deben eliminarse automáticamente, porque en contaminación pueden representar episodios reales; se recomienda revisarlos con histogramas y boxplots antes de decidir si se imputan, transforman o conservan.

In [ ]:
## ¿cómo son las correlaciones entre las variables del dataset?
corr_matrix = df_model.corr(numeric_only=True)
plt.figure(figsize=(12, 9))
plt.imshow(corr_matrix, aspect='auto')
plt.colorbar(label='Correlación de Pearson')
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=90)
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
plt.title('Matriz de correlación')
plt.tight_layout()
plt.show()
corr_matrix.round(3)


In [ ]:
# Correlaciones ordenadas respecto a la variable objetivo
target_correlations = corr_matrix[target].drop(target).sort_values(key=lambda s: s.abs(), ascending=False)
target_correlations


Las correlaciones muestran relaciones fuertes entre varias respuestas de sensores y concentraciones de contaminantes. Esto sugiere que existen variables con capacidad predictiva para estimar `C6H6(GT)`. Sin embargo, también puede existir multicolinealidad entre predictores, aspecto relevante para interpretar los coeficientes de la regresión lineal múltiple.

In [ ]:
## ¿qué tres variables son las más correlacionadas con la variable objetivo?
top3_corr = target_correlations.head(3)
top3_corr


Las tres variables más correlacionadas con la variable objetivo se obtienen con `top3_corr`. Para la regresión lineal simple se escogerá la primera de esa lista, porque es la variable individual que presenta mayor asociación lineal con `C6H6(GT)` dentro del análisis exploratorio.

In [ ]:
# Variable candidata para regresión lineal simple
simple_feature = top3_corr.index[0]
print('Variable seleccionada para regresión lineal simple:', simple_feature)
print('Correlación con la variable objetivo:', top3_corr.iloc[0])


In [ ]:
## ¿existe alguna variable que no tenga correlación?
low_corr = target_correlations[target_correlations.abs() < 0.10]
low_corr


Se consideran variables con correlación muy baja aquellas cuya correlación absoluta con `C6H6(GT)` es menor a 0.10. Si `low_corr` aparece vacío, no hay variables prácticamente desconectadas de la variable objetivo bajo este criterio; si aparece alguna, su aporte lineal directo sería limitado, aunque aún podría aportar en modelos no lineales como árboles de decisión.

In [ ]:
# Histogramas y boxplots para apoyar la detección visual de asimetría y valores atípicos
for col in pollutants + [target]:
    plt.figure(figsize=(6, 3))
    plt.hist(df_model[col].dropna(), bins=40)
    plt.title(f'Distribución de {col}')
    plt.xlabel(col)
    plt.ylabel('Frecuencia')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6, 2))
    plt.boxplot(df_model[col].dropna(), vert=False)
    plt.title(f'Boxplot de {col}')
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()


En base al EDA realizado, ¿qué suposiciones se pueden hacer sobre los datos?¿qué conclusiones extraes para implementar el modelo predictivo?

A partir del EDA se asume que el problema es de regresión supervisada con variables principalmente continuas, datos faltantes codificados como `-200`, asimetría positiva en contaminantes y posibles valores atípicos. Para implementar los modelos se requiere: reemplazar `-200`, eliminar registros sin variable objetivo, imputar predictores, separar entrenamiento/prueba, escalar las variables para regresión lineal y comparar modelos con métricas de error y bondad de ajuste. La regresión lineal simple servirá como línea base; la regresión múltiple aprovechará todas las variables; el árbol de decisión permitirá capturar relaciones no lineales.

### Regresión

Para llevar a cabo la tarea de regresión deseada se pretender hacer una comparativa de varios modelos. Unos usarán el algoritmo de regresión lineal, y otros realizarán la predicción haciendo uso de árboles de decisión.

Para los primeros modelos hay que usar el módulo https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html

El algoritmo de Regresión Lineal, necesita saber cuáles son las variables que va a tener en cuenta para realizar la estimación.

El primer modelo que se debe construir, usará una regresión lineal simple. Para ello sigue los siguientes pasos.

Antes de empezar con la implementación de los modelos hace falta realizar una transformación de datos, escalarlos.

In [ ]:
# Preparación de datos para los modelos
# 1. Se elimina la variable objetivo cuando está ausente.
# 2. Se separan predictores y respuesta.
# 3. Se realiza split train/test.
# 4. Se imputan nulos en X con la mediana calculada en entrenamiento.
# 5. Se escalan variables para regresión lineal.

model_data = df_model.dropna(subset=[target]).copy()
X = model_data.drop(columns=[target])
y = model_data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

imputer = SimpleImputer(strategy='median')
X_train_imp = DataFrame(imputer.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_imp = DataFrame(imputer.transform(X_test), columns=X.columns, index=X_test.index)

scaler = StandardScaler()
X_train_scaled = DataFrame(scaler.fit_transform(X_train_imp), columns=X.columns, index=X_train.index)
X_test_scaled = DataFrame(scaler.transform(X_test_imp), columns=X.columns, index=X_test.index)

print('X_train:', X_train_scaled.shape)
print('X_test:', X_test_scaled.shape)
print('y_train:', y_train.shape)
print('y_test:', y_test.shape)


In [ ]:
# separar datos de entrenamiento y test
print('Porcentaje entrenamiento:', round(len(X_train) / len(X) * 100, 2), '%')
print('Porcentaje prueba:', round(len(X_test) / len(X) * 100, 2), '%')


In [ ]:
# escoger la variable que a partir del EDA realizado, consideres que mejor va a realizar la predicción
simple_feature = top3_corr.index[0]
print('Variable seleccionada:', simple_feature)

X_train_simple = X_train_scaled[[simple_feature]]
X_test_simple = X_test_scaled[[simple_feature]]


In [ ]:
# entrena el modelo con los datos de entrenamiento
lr_simple = LinearRegression()
lr_simple.fit(X_train_simple, y_train)
print('Modelo de regresión lineal simple entrenado.')


In [ ]:
# ¿cuáles son los valores aprendidos por el modelo para los parámetros?
print('Intercepto:', lr_simple.intercept_)
print('Coeficiente para', simple_feature + ':', lr_simple.coef_[0])


Explica qué indican estos parámetros

El intercepto representa el valor esperado de `C6H6(GT)` cuando la variable predictora escalada toma valor cero, es decir, cuando está en su media después de la estandarización. El coeficiente indica cuánto cambia la concentración estimada de benceno por cada incremento de una desviación estándar en la variable predictora seleccionada. Si el coeficiente es positivo, la relación estimada es directa; si es negativo, la relación es inversa.

In [ ]:
# Función auxiliar para calcular métricas de regresión
def regression_metrics(y_true, y_pred):
    y_pred_non_negative = np.maximum(y_pred, 0)
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'R2': r2_score(y_true, y_pred),
        'RMSLE': np.sqrt(mean_squared_log_error(y_true, y_pred_non_negative))
    }


In [ ]:
# realiza las predicciones para el conjunto de datos de test
y_pred_simple = lr_simple.predict(X_test_simple)
pd.DataFrame({'real': y_test.values[:10], 'predicho': y_pred_simple[:10]})


In [ ]:
# Ahora es necesario evaluar el modelo. ¿Qué métrica es mejor utilizar en este caso? Justifica tu respuesta.
metrics_simple = regression_metrics(y_test, y_pred_simple)
metrics_simple


En este caso conviene usar principalmente MAE porque expresa el error promedio en las mismas unidades de la variable objetivo y es fácil de interpretar. También se reporta R² para medir la proporción de variabilidad explicada y RMSLE porque penaliza errores relativos, útil cuando hay asimetría y valores positivos. Para comparar todos los modelos se mantendrán las mismas métricas.

El error del modelo lineal simple corresponde al MAE calculado en la celda anterior. Este modelo usa una sola variable, por lo que funciona como línea base: si el MAE es alto o el R² es limitado, significa que una única relación lineal no captura toda la variabilidad de la concentración de benceno.

_indica aquí tu respuesta_

In [ ]:
# Gráfico de valores reales vs predichos para la regresión lineal simple
plt.figure(figsize=(5,5))
plt.scatter(y_test, y_pred_simple, alpha=0.4)
plt.xlabel('Valor real')
plt.ylabel('Valor predicho')
plt.title('Regresión lineal simple: real vs predicho')
plt.tight_layout()
plt.show()


Ahora debes entrenar un segundo modelo que haga uso de una regresión lineal múltiple con **todas las variables** del dataset. Después de entrenar, realiza las predicciones para este segundo modelo.

In [ ]:
# Regresión lineal múltiple con todas las variables predictoras
lr_multiple = LinearRegression()
lr_multiple.fit(X_train_scaled, y_train)
y_pred_multiple = lr_multiple.predict(X_test_scaled)
metrics_multiple = regression_metrics(y_test, y_pred_multiple)
metrics_multiple


In [ ]:
# Coeficientes del modelo múltiple
coef_multiple = pd.DataFrame({
    'variable': X_train_scaled.columns,
    'coeficiente': lr_multiple.coef_
}).sort_values('coeficiente', key=lambda s: s.abs(), ascending=False)
coef_multiple


In [ ]:
# Comparación inicial entre regresión lineal simple y múltiple
comparison_lr = pd.DataFrame([
    {'modelo': 'Regresión lineal simple', **metrics_simple},
    {'modelo': 'Regresión lineal múltiple', **metrics_multiple}
])
comparison_lr


¿Qué error tiene este modelo?¿Es mejor o peor que el anterior?

El error de la regresión lineal múltiple debe compararse con el de la regresión simple usando la tabla anterior. Si el MAE y el RMSLE disminuyen y el R² aumenta, el modelo múltiple es mejor porque aprovecha más información. Si la mejora es pequeña, puede deberse a multicolinealidad entre predictores o a que parte de la relación entre variables no es estrictamente lineal.

In [ ]:
# Gráfico de valores reales vs predichos para la regresión lineal múltiple
plt.figure(figsize=(5,5))
plt.scatter(y_test, y_pred_multiple, alpha=0.4)
plt.xlabel('Valor real')
plt.ylabel('Valor predicho')
plt.title('Regresión lineal múltiple: real vs predicho')
plt.tight_layout()
plt.show()


#### Regresión con árboles de decisión

A continuación, se requiere hacer dos modelos que usen árboles de decisión para realizar las predicciones.

Para los árboles de decisión, al ser una tarea de regresión, hay que usar el módulo https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeRegressor.html

El algoritmo **DTRegressor** necesita ajustar una serie de hiperparámetros para realizar las predicciones. La implementación de sklearn nos da mucha flexibilidad para nuestros modelos. En general, para los problemas más comunes de regresión, nos tenemos que preocupar de los siguientes hiperparámetros:

* `criterion`: función usada para medir la calidad de una partición. En regresión puede usar error cuadrático, error absoluto, Friedman MSE o Poisson.
* `splitter`: estrategia para escoger la división en cada nodo; puede ser la mejor división disponible (`best`) o una división aleatoria (`random`).
* `max_depth`: profundidad máxima del árbol. Controla la complejidad del modelo y ayuda a reducir sobreajuste.
* `min_samples_split`: número mínimo de muestras requeridas para dividir un nodo interno.
* `min_samples_leaf`: número mínimo de muestras que debe tener una hoja. Valores mayores suavizan el árbol.
* `max_features`: número de características consideradas al buscar la mejor división. Si es `None`, se usan todas.

El parámetro `min_impurity_decrease` indica la reducción mínima de impureza necesaria para aceptar una partición. Es útil para evitar divisiones que apenas mejoran el modelo, controlar el crecimiento del árbol y reducir overfitting.

Entrena un modelo de árboles de decisión donde, el criterio para realizar las particiones sea _poisson_, la profundidad máxima de los árboles debe ser 10, el número mínimo de ejemplos para realizar una partición debe ser 10, el número mínimo de ejemplos para considerarlo una hoja debe ser 2, y el número máximo de características deben ser todas.

In [ ]:
# Entrena un modelo de árboles de decisión con los hiperparámetros solicitados
dt_regressor = DecisionTreeRegressor(
    criterion='poisson',
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=2,
    max_features=None,
    random_state=RANDOM_STATE
)

# Para árboles no es obligatorio escalar; se usan datos imputados en la escala original.
dt_regressor.fit(X_train_imp, y_train)
y_pred_tree = dt_regressor.predict(X_test_imp)
print('Modelo DecisionTreeRegressor entrenado.')


In [ ]:
# Importancia de variables del árbol
feature_importance = pd.DataFrame({
    'variable': X_train_imp.columns,
    'importancia': dt_regressor.feature_importances_
}).sort_values('importancia', ascending=False)
feature_importance


Calcula MAE, R2 y RMSLE

In [ ]:
# Calcula MAE, R2 y RMSLE
metrics_tree = regression_metrics(y_test, y_pred_tree)
metrics_tree


Para comprobar si existe overfitting se deben comparar métricas en entrenamiento y prueba. Hay sobreajuste si el árbol obtiene R² muy alto y errores muy bajos en entrenamiento, pero su desempeño empeora claramente en prueba. Si ocurre, se puede reducir `max_depth`, aumentar `min_samples_split`, aumentar `min_samples_leaf` o usar validación cruzada para seleccionar hiperparámetros.

In [ ]:
# ¿Existe overfitting? Indica qué debes hacer para comprobar si hay overfitting.
y_pred_tree_train = dt_regressor.predict(X_train_imp)
metrics_tree_train = regression_metrics(y_train, y_pred_tree_train)

overfitting_check = pd.DataFrame([
    {'dataset': 'Entrenamiento', **metrics_tree_train},
    {'dataset': 'Prueba', **metrics_tree}
])
overfitting_check


¿Este modelo es mejor, peor o igual que los de regresión lineal simple y múltiple? Razona tu respuesta.

El árbol de decisión será mejor que los modelos lineales si presenta menor MAE/RMSLE y mayor R² en el conjunto de prueba. Su ventaja es que puede capturar relaciones no lineales e interacciones entre variables. Sin embargo, si la brecha entre entrenamiento y prueba es grande, el buen resultado en entrenamiento no sería confiable por posible overfitting.

**Comparativa**

En base al EDA realizado, a las decisiones tomadas sobre los datos e hiperparámetros y a las características computacionales de tu equipo, el mejor modelo será el que tenga menor MAE y RMSLE, junto con mayor R² en prueba. En términos metodológicos, la regresión lineal simple se usa como referencia mínima, la regresión múltiple evalúa el aporte conjunto de todas las variables y el árbol de decisión evalúa relaciones no lineales. La conclusión final debe apoyarse en la tabla `model_comparison` generada abajo.

In [ ]:
# Comparativa final de modelos
model_comparison = pd.DataFrame([
    {'modelo': 'Regresión lineal simple', **metrics_simple},
    {'modelo': 'Regresión lineal múltiple', **metrics_multiple},
    {'modelo': 'Árbol de decisión Poisson', **metrics_tree}
]).sort_values(['MAE', 'RMSLE'])
model_comparison


In [ ]:
# Visualización comparativa
plt.figure(figsize=(8, 4))
plt.bar(model_comparison['modelo'], model_comparison['MAE'])
plt.ylabel('MAE')
plt.title('Comparación de MAE por modelo')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

best_model = model_comparison.iloc[0]['modelo']
print('Mejor modelo según MAE:', best_model)


## Investigación

**Referencia APA del artículo seleccionado**

Bhati, V. S., Saxena, A., & Khatwal, R. (2024). *Exploring air quality dynamics and predictive modeling by using artificial intelligence during COVID-19 lock down over the western part of India*. **Current World Environment, 19**(2). https://doi.org/10.12944/CWE.19.2.36

**Objetivo de la investigación**

El artículo aplica modelos supervisados para analizar y predecir la calidad del aire en ciudades del oeste de India durante periodos preconfinamiento, confinamiento y posconfinamiento por COVID-19. El problema real es estimar el comportamiento del índice de calidad del aire y contaminantes asociados para apoyar la toma de decisiones ambientales.

**Técnicas de regresión empleadas**

El estudio compara modelos de regresión como regresión lineal, árbol de decisión, random forest y XGBoost. Para esta actividad son especialmente relevantes la regresión lineal y el árbol de decisión, porque coinciden con las técnicas trabajadas en el laboratorio. No se plantea una arquitectura de deep learning, sino modelos clásicos de aprendizaje automático supervisado.

**Principales resultados**

Los autores reportan que los modelos basados en árboles y ensambles tuvieron mejor desempeño que la regresión lineal en la predicción de AQI. En particular, la comparación muestra que el árbol de decisión puede ajustar relaciones no lineales entre contaminantes y calidad del aire, aunque también puede presentar diferencias entre entrenamiento y prueba si no se controla su complejidad. El trabajo concluye que estos modelos pueden ser útiles para monitoreo ambiental y apoyo a políticas públicas de control de contaminación.

In [ ]:
# Declaración ética de apoyo con IA generativa
print('Declaración: se utilizó IA generativa como apoyo para estructurar explicaciones, revisar código y mejorar redacción. La ejecución, validación, interpretación final y entrega son responsabilidad del estudiante.')
